# Epithelial-centered Neighborhood Profiling 

To characterize the cellular composition surrounding each epithelial cell, we defined a neighborhood as all cells located within a radius of 80 µm. For each epithelial cell, we counted the number of neighboring cells belonging to each subtype within this neighborhood. The resulting counts (epithelial cells × cell types) were used for downstream analyses such as PCA, clustering, and niche classification.

In [ ]:
import scanpy as sc
import matplotlib
import matplotlib.pyplot as plt
import os
import numpy as np
import squidpy as sq
import anndata
import pandas as pd
import seaborn as sns
import tqdm
from Xenium_5K_IPF_constants import ALL_SUBTYPE_LIST, SAMPLES_SYMBOLS, WORK_PATH

# Option
sc.settings.verbosity = 3
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl

# Constant Declaration
distance = 80
min_threshold = 5

In [ ]:
adata = sc.read_h5ad(f"{WORK_PATH}AnnData/spatial_merged.h5ad")

In [ ]:
celltype_csv = pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv", index_col="cell_id")
adata = adata[celltype_csv.index].copy()
adata.obs = pd.merge(adata.obs, celltype_csv[["cell_type", "subtype", "sample"]], left_index=True, right_index=True)

In [ ]:
save_name = f"{WORK_PATH}CSV/021_EcNP_score_{distance}.csv"
sub_adata = adata[adata.obs["cell_type"] == "Epithelial"].copy()
coords = adata.obsm["spatial"].copy()

In [ ]:
score_list = []
hoge = False
if os.path.isfile(save_name):
    score_df=pd.read_csv(save_name, index_col=0)
    hoge = True

if hoge:
    print("Already done!")

else:
    for idx in tqdm.tqdm(range(len(sub_adata))):
        # Obtain the coordinates of the central cell
        target_coord = sub_adata.obsm["spatial"][idx]
    
        # Distance calculation
        distances = np.linalg.norm(coords - target_coord, axis=1)
        
        # Neighbor cells within a certain distance
        mask = (distances <= distance) 
        neighbor_indices = np.where(mask)[0]
    
        neighbor_distances = distances[neighbor_indices]
        neighbor_subtypes = adata.obs["subtype"].values[neighbor_indices]
        
        # Score calculation for each subtype
        subtype_score = pd.Series(data=neighbor_subtypes).value_counts()
        
        # Scores for all subtypes
        full_score = {sub: 0.0 for sub in ALL_SUBTYPE_LIST}
        full_score.update(subtype_score.to_dict())
    
        score_list.append(full_score)
    # Output（Dataframe: epithelial subtypes × all subtypes）
    score_df = pd.DataFrame(score_list, index=sub_adata.obs_names)
    
    score_df.to_csv(save_name, encoding="utf-8-sig")


In [ ]:
score_adata = anndata.AnnData(
    X=score_df.values.copy(),
    obs=sub_adata.obs.copy(),
    obsm={"spatial": sub_adata.obsm["spatial"].copy()},
    var=pd.DataFrame(index=score_df.columns)
)
score_adata.raw = score_adata.copy()

In [ ]:
count_df = pd.DataFrame([sub_adata.obs["sample"],score_df.sum(axis=1)], index=["sample", "counts"]).T

In [ ]:
count_df["sample"] = pd.Categorical(count_df["sample"], categories=SAMPLES_SYMBOLS)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2,1,figsize=(4,6), height_ratios=(1.1,2))
sample_col = "grey"
sns.barplot(pd.DataFrame(count_df["sample"].value_counts()), x="sample", y="count", hue="sample", ax=ax1, palette=sample_col, edgecolor="black", alpha=0.8)
ax1.set_xticks(ticks=ax1.get_xticks(), labels=[])
ax1.set_xlabel("")
ax1.set_yticks(ticks=[0, 1e4, 2e4], labels=["0", "10K", "20K"])
ax1.set_ylabel("Number of\nepithelial cells", fontsize=14)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.tick_params("y", labelsize=13)

sns.violinplot(count_df, x="sample", y="counts", inner=None, ax=ax2, color="grey", bw_adjust=2, hue="sample",palette=sample_col, alpha=0.8)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.set_xlabel("")
ax2.set_xticks(ticks=ax2.get_xticks(), labels=ax2.get_xticklabels(), fontsize=15, rotation=45, rotation_mode='anchor', ha="right", va="top")
ax2.tick_params("y", labelsize=13)
ax2.set_ylabel("Number of neighboring cells\nper epithelial cell", fontsize=14)
ax2.set_ylim(0,450)
fig.tight_layout()
fig.savefig(f"{WORK_PATH}Figure/021_neighbor_cell_counts.png", dpi=300)

In [ ]:
sc.pp.filter_cells(score_adata, min_counts=min_threshold)

In [ ]:
sc.pp.normalize_total(score_adata, target_sum=1e4)
sc.pp.log1p(score_adata)

In [ ]:
sc.pp.pca(score_adata, n_comps=5)
sc.pp.neighbors(score_adata, n_neighbors=50, use_rep="X_pca")
sc.tl.umap(score_adata, min_dist=0.4, spread=0.6)
sc.tl.leiden(score_adata, flavor="igraph", directed=False, resolution=0.25)

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
sc.pl.umap(score_adata, color="leiden", legend_loc="on data", s=3, ax=ax, legend_fontoutline=3, legend_fontsize=30, show=False)

plt.title("")
ax.axis("off")

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_umap_leiden.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
fig, ax = plt.subplots(figsize=(20/0.9, 11/1))
fig.subplots_adjust(left=0, right=0.9, bottom=0, top=1)
fig.set_facecolor("none")
sq.pl.spatial_scatter(score_adata, library_id="spatial", size=2, shape=None, color="leiden", ax=ax)
plt.title("", fontsize=10)
ax.legend(labelcolor="black",
          markerscale=4,
          facecolor="none",
          borderpad=1,
          edgecolor="none",
          loc="center left",
          bbox_to_anchor=(0.98,0.5),
          handletextpad=0.4,
          labelspacing=0.5,
          fontsize=36,
         )
ax.axis("off")

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_spatial_leiden_.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
score_df = score_df.loc[score_adata.obs_names]
score_df["cluster"] = score_adata.obs["leiden"].values
grouped = score_df.groupby("cluster")

mean_expr = grouped.mean()
presence_ratio = grouped.apply(lambda x: (x > 0).sum() / len(x))

mean_expr_norm = mean_expr / mean_expr.max()
presence_ratio = presence_ratio.loc[mean_expr_norm.index]

celltypes = list(score_df["cluster"].cat.categories)

mean_expr_plot = mean_expr_norm.loc[celltypes, ALL_SUBTYPE_LIST]
presence_plot = presence_ratio.loc[celltypes, ALL_SUBTYPE_LIST]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.2))
ax.set_position([0.04, 0.48, 0.95, 0.51])

l_cls = len(score_adata.obs["leiden"].unique())
cls_ax=fig.add_axes([0.032, 0.48, 0.008, 0.51], zorder=1)
cls_ax.set_xlim(0,1)
cls_ax.set_ylim(l_cls, 0)
cls_ax.axis("off")
for ys, c in zip(range(l_cls), score_adata.uns["leiden_colors"]):
    r = matplotlib.patches.Rectangle(xy=(0,ys) , width=1, height=1, fc=c)
    cls_ax.add_patch(r)

for i, celltype in enumerate(celltypes):
    for j, subtype in enumerate(ALL_SUBTYPE_LIST):
        size = presence_plot.loc[celltype, subtype] * 200
        color = mean_expr_plot.loc[celltype, subtype]
        ax.scatter(j, i, s=size, c=[plt.cm.viridis(color)], edgecolors='gray')

ax.set_xticks(range(len(ALL_SUBTYPE_LIST)))
ax.set_xticklabels(ALL_SUBTYPE_LIST, rotation=90)
ax.set_yticks(range(len(celltypes)))
ax.set_yticklabels(celltypes)
ax.set_xlim(-0.5,len(ALL_SUBTYPE_LIST)-0.5)
ax.set_ylim(l_cls-0.5, -0.5)
ax.tick_params("y", size=10, labelsize=14)
ax.tick_params("x", labelsize=14)

# Color bar legened
norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
c_ax = fig.add_axes([0.68,0.08,0.13,0.04])
cbar = fig.colorbar(sm, cax=c_ax, orientation='horizontal', pad=0.05)
cbar.set_ticks([0,0.5,1])
c_ax.set_title("Normalized mean count\n(per subtype)", fontsize=12)
c_ax.tick_params(labelsize=12)

# Dot size legend
d_ax = fig.add_axes([0.86, 0.01, 0.13, 0.11])  # [left, bottom, width, height]
d_ax.axis("off")
d_ax.set_xlim(0.2, 0.9)
d_ax.set_ylim(0.18, 0.206)
for frac, xpos in zip([20, 40, 60, 100], [0.24, 0.42, 0.61, 0.8]):
    d_ax.scatter(xpos, 0.2, s=frac * 2, color="lightgray", edgecolors="gray")
    d_ax.text(xpos, 0.185, f"{frac}", ha="center", fontsize=12)

d_ax.set_title("Detection rate\n(% cells > 0)", fontsize=12)
d_ax.tick_params(labelsize=12)

plt.show()

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_dot_leiden.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
# Niche annotation
niche_type = {
    "Near-normal alveolar": [5],
    "Pre-fibrotic": [2,4],
    "Fibrotic": [0,1,9],
    "Metaplastic": [6, 10],
    "Bronchiolar": [7,8],
    "Alveolar duct": [3],
    "Epithelial-rich": [11]
}

niche_lei2key = {}
for ni, le in niche_type.items():
    for l in le:
        niche_lei2key[str(l)] = ni
        
niche_lei2key_sort = {}
for l in range(len(score_adata.obs["leiden"].unique())):
    niche_lei2key_sort[str(l)] = niche_lei2key[str(l)]

score_adata.obs['EcNP'] = score_adata.obs['leiden'].map(niche_lei2key_sort).astype('category')
score_adata.obs['EcNP'] = pd.Categorical(values=score_adata.obs['EcNP'], categories=list(niche_type), ordered=True)

In [ ]:
score_adata.uns["EcNP_colors"] = [ '#466ebb', '#d76411', '#50ba9a', '#eaaa30', '#01abe9', '#ae4d91', '#e387b2']

In [ ]:
fig, ax = plt.subplots(figsize=(3/0.55,3))
fig.subplots_adjust(left=0, right=0.55, top=1, bottom=0)
sc.pl.umap(score_adata, color="EcNP", s=2, ax=ax)
for obj in ax.get_children():
    if hasattr(obj, "get_text") and obj.get_text() in {"0", "2"}:
        obj.set_fontsize(16)
plt.title("")
plt.axis("off")
ax.legend(title="", loc='upper left', bbox_to_anchor=(1, 0.98), edgecolor="white", labelcolor="black", markerscale=1.5, fontsize=13, title_fontsize=13, handletextpad=0.05)
plt.savefig(f"{WORK_PATH}Figure/021_EcNP_umap_cluster.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
sc.pl.umap(score_adata, color="EcNP", legend_loc="none", s=2, ax=ax, show=False)

plt.title("")
ax.axis("off")

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_umap_nolegend.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
fig, ax = plt.subplots(figsize=(20/0.7, 11/1))
fig.subplots_adjust(left=0, right=0.7, bottom=0, top=1)
fig.set_facecolor("none")
sq.pl.spatial_scatter(score_adata, library_id="spatial", size=2, shape=None, color="EcNP", ax=ax)
plt.title("", fontsize=10)
ax.legend(labelcolor="black",
          markerscale=4,
          facecolor="none",
          borderpad=1,
          edgecolor="none",
          loc="center left",
          bbox_to_anchor=(0.98,0.5),
          handletextpad=0.4,
          labelspacing=0.5,
          fontsize=36,
         )
ax.axis("off")

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_spatial_cluster.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
score_df = score_df.loc[score_adata.obs_names]
score_df["cluster"] = score_adata.obs["EcNP"].values
grouped = score_df.groupby("cluster")

mean_expr = grouped.mean()
presence_ratio = grouped.apply(lambda x: (x > 0).sum() / len(x))

mean_expr_norm = mean_expr / mean_expr.max()
presence_ratio = presence_ratio.loc[mean_expr_norm.index]

celltypes = list(score_df["cluster"].cat.categories)

mean_expr_plot = mean_expr_norm.loc[celltypes, ALL_SUBTYPE_LIST]
presence_plot = presence_ratio.loc[celltypes, ALL_SUBTYPE_LIST]

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.set_position([0.2, 0.48, 0.79, 0.51])

l_cls = len(score_adata.obs["EcNP"].unique())
cls_ax=fig.add_axes([0.19, 0.48, 0.01, 0.51], zorder=1)
cls_ax.set_xlim(0,1)
cls_ax.set_ylim(l_cls, 0)
cls_ax.axis("off")
for ys, c in zip(range(l_cls), score_adata.uns["EcNP_colors"]):
    r = matplotlib.patches.Rectangle(xy=(0,ys) , width=1, height=1, fc=c)
    cls_ax.add_patch(r)

for i, celltype in enumerate(celltypes):
    for j, subtype in enumerate(ALL_SUBTYPE_LIST):
        size = presence_plot.loc[celltype, subtype] * 200
        color = mean_expr_plot.loc[celltype, subtype]
        ax.scatter(j, i, s=size, c=[plt.cm.viridis(color)], edgecolors='gray')

ax.set_xticks(range(len(ALL_SUBTYPE_LIST)))
ax.set_xticklabels(ALL_SUBTYPE_LIST, rotation=90)
ax.set_yticks(range(len(celltypes)))
ax.set_yticklabels(celltypes)
ax.set_xlim(-0.5,len(ALL_SUBTYPE_LIST)-0.5)
ax.set_ylim(l_cls-0.5, -0.5)
ax.tick_params("y", size=15, labelsize=15)
ax.tick_params("x", labelsize=14)

# Color bar legened
norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
c_ax = fig.add_axes([0.73,0.08,0.11,0.04])
cbar = fig.colorbar(sm, cax=c_ax, orientation='horizontal', pad=0.05)
cbar.set_ticks([0,0.5,1])
c_ax.set_title("Normalized mean count\n(per subtype)", fontsize=12)
c_ax.tick_params(labelsize=12)

# Dot size legend
d_ax = fig.add_axes([0.88, 0.01, 0.11, 0.11])  # [left, bottom, width, height]
d_ax.axis("off")
d_ax.set_xlim(0.2, 0.9)
d_ax.set_ylim(0.18, 0.206)
for frac, xpos in zip([20, 40, 60, 100], [0.24, 0.42, 0.61, 0.8]):
    d_ax.scatter(xpos, 0.2, s=frac * 2, color="lightgray", edgecolors="gray")
    d_ax.text(xpos, 0.185, f"{frac}", ha="center", fontsize=12)

d_ax.set_title("Detection rate\n(% cells > 0)", fontsize=12)
d_ax.tick_params(labelsize=12)

plt.show()

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_dot_cluster.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
score_adata.write(f"{WORK_PATH}AnnData/EcNP.h5ad")

In [ ]:
celltype_csv = pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv", index_col="cell_id")

In [ ]:
celltype_c = celltype_csv.loc[score_adata.obs_names,]

In [ ]:
# Stacked bar of subtype proportion
ct_counts = pd.crosstab(score_adata.obs['sample'], score_adata.obs['EcNP'])
ct_props = ct_counts.div(ct_counts.sum(axis=1), axis=0)

cell_type_order = ct_counts.columns.tolist()

ct_props = ct_props[cell_type_order[::-1]] 

colors = score_adata.uns["EcNP_colors"]
colors_for_plot = colors[::-1]

ax = ct_props.plot(kind='bar', stacked=True, figsize=(7, 4), color=colors_for_plot)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], title='', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=15, edgecolor="none")

plt.ylabel(f'Profile proportion', fontsize=15)
plt.xlabel('')
plt.xticks(size=15, rotation=45, rotation_mode='anchor', ha="right", va="top")
plt.tick_params("y", labelsize=13)
plt.subplots_adjust(left=0.1, right=0.45, bottom=0.18, top=0.98)

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_stackedbar.png", dpi=500, format="png", bbox_inches=None)
plt.show()

# Pseudo-temporal Continuum Analysis based on Epithelial Niches

In [ ]:
import numpy as np
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import matplotlib
import scicm
import pandas as pd
import scFates as scf
import seaborn as sns
from Xenium_5K_IPF_constants import ALL_SUBTYPE_LIST, TITLE_ABBREVIATION, EPITHELIAL_SUBTYPE, CELL_TYPE_3M, WORK_PATH

# Option
sc.settings.verbosity = 3
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl

# Constant Declaration
p_time_cmap = "plasma"

In [ ]:
n_adata = sc.read_h5ad(f"{WORK_PATH}AnnData/EcNP.h5ad")
n_adata.obs["subtype"]=pd.Categorical(n_adata.obs["subtype"], categories=EPITHELIAL_SUBTYPE, ordered=True)

In [ ]:
# Epithelial subtype distribution on UMAP of niche analysis 
ht_size = 5
fig, ax=plt.subplots(figsize=(ht_size/(10/7), ht_size))
sc.pl.umap(n_adata, color="subtype", ax=ax, s=4,show=False)
ax.legend(loc="upper center",bbox_to_anchor=(0.48,0.05), edgecolor="white", fontsize=14, labelspacing=0.3, handletextpad=0.1, ncols=2, markerscale=1.5)
ax.axis("off")
ax.set_title("Epithelial subtype",fontsize=15)
plt.subplots_adjust(left=0, right=1, bottom=0.22, top=0.92)
plt.savefig(f"{WORK_PATH}Figure/021_EcNP_umap_epi_all.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
# by subtype
fig, axes = plt.subplots(2,4, figsize=(ht_size*0.92*4.15/2.2, ht_size))
axes = axes.flatten()
for ax,s in zip(axes, EPITHELIAL_SUBTYPE):
    sc.pl.umap(n_adata, color="subtype", groups=s, legend_loc=None, ax=ax, show=False, s=3)
    ax.axis("off")
    ax.set_title(s, fontsize="15")
plt.subplots_adjust(left=0, right=1, bottom=0, top=0.92, wspace=0.05, hspace=0.2)
axes[-1].axis("off")

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_umap_epi_sub.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
n_adata = n_adata[~n_adata.obs["EcNP"].isin(['Bronchiolar', 'Alveolar duct', "Epithelial-rich"])]

In [ ]:
# scFate
scf.tl.curve(n_adata,Nodes=30,use_rep="X_pca",ndims_rep=2,epg_lambda=0.1, epg_mu=1)

In [ ]:
fig, ax = plt.subplots(figsize=(3.5/0.6,3))
fig.subplots_adjust(left=0, right=0.6, top=1, bottom=0)
scf.pl.graph(n_adata,basis="umap", color_cells="EcNP", s=2, ax=ax)
for obj in ax.get_children():
    if hasattr(obj, "get_text") and obj.get_text() in {"0", "2"}:
        obj.set_fontsize(16)
plt.title("")
plt.axis("off")
ax.legend(title="", loc='upper left', bbox_to_anchor=(0.92, 0.98), edgecolor="white", labelcolor="black", markerscale=1.5, fontsize=13, title_fontsize=13, handletextpad=0.05)
plt.savefig(f"{WORK_PATH}Figure/021_EcNP_umap_cPosition_tree.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
scf.tl.root(n_adata,0)

In [ ]:
scf.tl.pseudotime(n_adata,n_jobs=20,n_map=20,seed=1)

In [ ]:
# max_min scaling
t = n_adata.obs["t"].values
n_adata.obs["t"] = (t - np.min(t)) / (np.max(t) - np.min(t))

In [ ]:
fig, ax = plt.subplots(figsize=(3*3.5/0.6,3*3))
fig.subplots_adjust(left=0, right=0.6, top=1, bottom=0)
scf.pl.trajectory(n_adata,basis="umap", arrows=True, arrow_offset=1, scale_path=3, color_cells="EcNP", ax=ax, s=8, cmap_seg=p_time_cmap, )
plt.title("")
plt.axis("off")
ax.legend(title="", loc='upper left', bbox_to_anchor=(0.92, 0.95), edgecolor="white", labelcolor="black", markerscale=5, fontsize=36, handletextpad=0.05)
values = n_adata.obs["t"]
norm = matplotlib.colors.Normalize(vmin=values.min(), vmax=values.max())
sm = plt.cm.ScalarMappable(cmap=p_time_cmap, norm=norm)
cbar_ax = fig.add_axes([0.6, 0.15, 0.25, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, ticks=[0.0, 0.5, 1.0], orientation="horizontal")
cbar.ax.set_xticklabels(['0.0', "0.5", "1.0"], fontsize=26)
cbar.ax.set_title("Continuum position", fontsize=30)
ax.get_children()[6].headwidth=ax.get_children()[7].headwidth
ax.get_children()[6].scale=ax.get_children()[6].scale*0.82
ax.get_children()[6].set_offsets([[ax.get_children()[6].get_offsets()[0][0]-0.05,ax.get_children()[6].get_offsets()[0][1]-0.01]])
ax.get_children()[8].set(sizes=np.array([2000]))
ax.get_children()[10].set(sizes=np.array([1400]))
fig.savefig(f"{WORK_PATH}Figure/021_EcNP_umap_cPosition_trajectory.png", dpi=100, format="png", bbox_inches=None)

In [ ]:
fig, ax = plt.subplots(figsize=(3.5/0.7,3))
fig.subplots_adjust(left=0, right=0.7, bottom=0, top=1)
sc.pl.umap(n_adata, color="t", cmap=p_time_cmap, s=5, outline_width=(0,0), add_outline=True, ax=ax, colorbar_loc=None, show=False)
ax.axis("off")
ax.set_title("")

values = n_adata.obs["t"]
norm = matplotlib.colors.Normalize(vmin=values.min(), vmax=values.max())
sm = plt.cm.ScalarMappable(cmap=p_time_cmap, norm=norm)
cbar_ax = fig.add_axes([0.75, 0.15, 0.21, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, ticks=[0.0, 0.5, 1.0], orientation="horizontal")
cbar.ax.set_xticklabels(['0.0', "0.5", "1.0"], fontsize=12)
cbar.ax.set_title("Continuum\nposition", fontsize=13)
plt.show()

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_umap_cPosition_gradation.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
tmp_title= [TITLE_ABBREVIATION.get(element, element)  for element in ALL_SUBTYPE_LIST]

In [ ]:
# Color scale is independently normalized within each subtype (min–max per panel).

fig=plt.figure(figsize=(9.5,7))
axes = [fig.add_subplot(5, 8, 1+n) for n in range(len(ALL_SUBTYPE_LIST))]

for sb, ax, ti in zip(ALL_SUBTYPE_LIST, axes, tmp_title):
    sc.pl.umap(n_adata, color=sb, cmap="scicm.R2O", s=1, use_raw=True, title="", add_outline=True, ax=ax, colorbar_loc=None, show=False)
    ax.set_title(ti, fontsize=12)
    ax.axis("off")

norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap="scicm.R2O", norm=norm)
cbar_ax = fig.add_axes([7/8, 0.06, 1/8*0.9, 0.025])

cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_ticks([])
cbar_ax.set_title("Within-subtype\nscaled count", fontsize=12)

cbar_ax.text(0.0, -0.3, "min", transform=cbar_ax.transAxes, ha="left", va="top", fontsize=11)
cbar_ax.text(1.0, -0.3, "max", transform=cbar_ax.transAxes, ha="right", va="top", fontsize=11)

plt.subplots_adjust(left=0.01, right=0.98, bottom=0.02, top=0.94, hspace=0.4, wspace=0.15)
plt.show()

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_abundance_umap_cPosition.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
fig, ax = plt.subplots(figsize=(20/1, 10/0.9))
fig.subplots_adjust(left=0, right=1, bottom=0.1, top=1)
fig.set_facecolor("none")
sq.pl.spatial_scatter(n_adata, library_id="spatial", title="", size=2, shape=None, color="t", ax=ax, cmap=p_time_cmap, colorbar=None)
ax.axis("off")

norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap=p_time_cmap, norm=norm)
cbar_ax = fig.add_axes([0.78, 0.08, 0.16, 0.03])
cbar = fig.colorbar(sm, cax=cbar_ax, ticks=[0, 0.5, 1], orientation="horizontal")
cbar.ax.set_xticklabels(['0.0', "0.5", "1.0"], fontsize=30, c="black")
cbar_ax.set_title("Continuum position", fontsize=30, c="black")
cbar_ax.tick_params("x", color="black")

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_spatial_cPosition.png", dpi=500, format="png", bbox_inches=None)

plt.show()

In [ ]:
adata=n_adata.raw.to_adata()

In [ ]:
scf.tl.test_association(adata,n_jobs=20, fdr_cut=0.1, A_cut=0.1)

In [ ]:
scf.tl.fit(adata,n_jobs=20)

In [ ]:
fig=plt.figure(figsize=(10.5,7))
axes = [fig.add_subplot(5, 8, 1+n) for n in range(len(ALL_SUBTYPE_LIST))]

for i, (t, ax, ti) in enumerate(zip(ALL_SUBTYPE_LIST, axes, tmp_title)):
    scf.pl.single_trend(adata, t, basis="umap", color_exp="k", alpha_expr=0.003, size_expr=1, ax_trend=ax, plot_emb=False, show=False)

    ydata = []
    for child in ax.get_children():
        if isinstance(child, matplotlib.collections.PathCollection):
            offsets = child.get_offsets()
            ydata.extend(offsets[:, 1])
    ax.set_ylim(bottom=min(-np.percentile(ydata, 99)/8, -1), top=max(np.percentile(ydata, 99.5),5))
    
    ax.get_children()[1].set(color="red")
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.set_xticks([0.0, 0.5, 1.0])
    ax.set_title(ti, fontsize=12)
    ax.tick_params("x", labelsize=10)
    ax.tick_params("y", labelsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

leg_ax = fig.add_subplot([7/8,0.05, 1/8, 0.15])
leg_ax.annotate('', xy=[0.95,0.2], xytext=[0.15,0.2],
            arrowprops=dict(shrink=0, width=1.5, headwidth=7, 
                        headlength=7, connectionstyle='arc3',
                        facecolor='black', edgecolor='black')
           )
           
leg_ax.annotate('', xy=[0.15,0.85], xytext=[0.15,0.2],
            arrowprops=dict(shrink=0, width=1.5, headwidth=7, 
                        headlength=7, connectionstyle='arc3',
                        facecolor='black', edgecolor='black')
           )
leg_ax.text(x=0.16, y=0.17, s="Continuum\nposition", va="top", ma="center", size=12)
leg_ax.text(x=0.01, y=0.24, s="Count",va="bottom", size=12, rotation=90)
leg_ax.axis("off")


plt.subplots_adjust(left=0.03, right=0.98, bottom=0.05, top=0.96, hspace=1, wspace=0.4)

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_abundance_line_cPositon.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
obs = n_adata.obs[["subtype", "EcNP", "t", "sample"]].copy()
obs.rename(columns={"t":"C_position"},inplace=True)
t_csv = pd.DataFrame({"cell_id":n_adata.obs_names}).set_index("cell_id", drop=False)
t_csv = pd.merge(t_csv,obs, left_index=True, right_index=True).set_index("cell_id")
t_csv.to_csv(f"{WORK_PATH}CSV/021_epithelial_cPosition.csv", index="cell_id", encoding="utf-8-sig")


In [ ]:
sample_sort=list(t_csv.groupby("sample", observed=True)["C_position"].mean().sort_values().index)
t_csv["sample"]=pd.Categorical(t_csv["sample"], categories=sample_sort, ordered=True)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    data=t_csv,
    x="sample",
    y="C_position",
    inner="quart", 
    cut=0,
    linewidth=1,
    color="white",
    ax=ax,
    width=0.8,
    density_norm='count',
    bw_adjust=1.5,
    zorder=3
)

for n,line in enumerate(ax.lines):
    if n % 3 == 1:
        line.set(linestyle="-", color="white", lw=2, zorder=2)
    else:
        line.set(linestyle=(0,(2,1)), color="white", lw=1.5, zorder=2)

orig_xlim = ax.get_xlim()

Ny = 512
grad = np.linspace(0, 1, Ny).reshape(Ny, 1)

cmap = matplotlib.cm.plasma
norm = matplotlib.colors.Normalize(vmin=0, vmax=1)

violin_bodies = [
    c for c in ax.collections
    if isinstance(c, matplotlib.collections.PolyCollection) and len(c.get_paths()) > 0
]
ax.autoscale(False)

for body in violin_bodies:
    path_b = body.get_paths()[0]
    verts = path_b.vertices

    x0, x1 = verts[:, 0].min(), verts[:, 0].max()

    body.set_facecolor("none")
    body.set_edgecolor("black")
    body.set_linewidth(1.0)

    clip = matplotlib.patches.PathPatch(matplotlib.path.Path(verts), transform=ax.transData)

    ax.imshow(
        grad,
        origin="lower",
        aspect="auto",
        extent=[x0, x1, 0, 1],
        cmap=cmap,
        norm=norm,
        clip_path=clip,
        clip_on=True,
        zorder=1
    )

ax.set_xlim(orig_xlim)
ax.set_ylim(0,1)
ax.set_xlabel("")
ax.set_ylabel("Continuum position", fontsize=15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params("both", labelsize=15)
plt.tight_layout()


plt.savefig(f"{WORK_PATH}Figure/021_Cposition_sample_violin.png", dpi=500, format="png", bbox_inches=None)
plt.show()

# Pseudo-temporal Continuum Enrichment Anlysis

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LinearSegmentedColormap
import scicm
import pandas as pd
import decoupler as dc
import squidpy as sq
from Xenium_5K_IPF_constants import CMAP_COLORLIST, WORK_PATH

# Option
sc.settings.verbosity = 3
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl

In [ ]:
odata = sc.read_h5ad(f"{WORK_PATH}AnnData/integrated_anndata.h5ad")
n_adata = sc.read_h5ad(f"{WORK_PATH}AnnData/EcNP.h5ad")

n_adata = n_adata[~n_adata.obs["EcNP"].isin(['Bronchiolar', 'Alveolar duct', "Epithelial-rich"])]

pseudotime_df=pd.read_csv(f"{WORK_PATH}CSV/021_epithelial_cPosition.csv", index_col="cell_id")

In [ ]:
adata = odata[pseudotime_df.index,]
adata.obs = pd.merge(n_adata.obs, pseudotime_df[["C_position"]], left_index=True, right_index=True)
adata.obsm=n_adata.obsm.copy()
adata.raw = adata.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
progeny = dc.op.progeny(organism="human")

In [ ]:
adata.uns["EcNP_colors"]=n_adata.uns["EcNP_colors"]

In [ ]:
dc.mt.mlm(data=adata, net=progeny)

In [ ]:
score = dc.pp.get_obsm(adata=adata, key="score_mlm")

In [ ]:
pws = dc.tl.rankby_order(adata=score, order='C_position')
bin_pws = dc.pp.bin_order(
    adata=score.copy(),
    order="C_position",
    names=pws["name"].to_list(),
    label="EcNP",
    nbins=20
)

In [ ]:
fig, ax = plt.subplots(figsize=(6,6.2))
ax_x0, ax_x1 = 0.4, 0.96
ax_w = ax_x1 - ax_x0
fig.subplots_adjust(left=ax_x0, right=ax_x1, top=0.95, bottom=0.15)
dc.pl.order(
    df=bin_pws,
    mode='mat',
    kw_order=dict(vmin=-2, vmax=2, cmap='scicm.BkO'),
    ax=ax,
)

# delete X-axis labels
ax.set_xlabel("")
ax.set_xticks([])

ax.set_ylim(0,14)

fig.get_children()[2].set_visible(False)

ax.tick_params("y", labelsize=16)

# profile color bar
ax_n = fig.add_axes([ax_x0, 0.82, ax_w, 0.06])
dc.pl.order(
    df=bin_pws,
    mode='mat',
    ax=ax_n,
)
ax_n.set_ylim(14, 15.4)
fig.get_children()[4].set_visible(False)
ax_n.set_xlabel("")
ax_n.set_xticks([])
ax_n.set_yticks([14.7])
ax_n.set_yticklabels(["Neighborhood class"], fontsize=16, weight="bold")

# Pseudo-temporal continuum color bar

norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap="plasma", norm=norm)

ax_c = fig.add_axes([ax_x0, 0.89, ax_w, 0.05])
cbar = fig.colorbar(sm, cax=ax_c, ticks=[0, 0.5, 1], orientation="horizontal")
cbar.ax.set_xticklabels(['0.0', "0.5", "1.0"], fontsize=14, c="black")
ax_c.xaxis.set_ticks_position('top')
ax_c.set_yticks([0.5])
ax_c.set_yticklabels(["Continuum position"], fontsize=16, weight="bold")


cbar_ax = fig.add_axes([ax_x0+0.1, 0.05, ax_w-0.2, 0.05])
norm_cb = matplotlib.colors.Normalize(vmin=-1.7, vmax=1.7)
sm_cb = plt.cm.ScalarMappable(cmap="scicm.BkO", norm=norm_cb)
cbar_cb = fig.colorbar(sm_cb, cax=cbar_ax, orientation="horizontal")
cbar_ax.set_title("Scaled pathway activity", fontsize=15)
cbar_ax.tick_params("x", labelsize=14)

label_ax = plt.gcf().add_axes([0.1, ax.get_position().y0, 0.1, ax.get_position().height], zorder=-1)
label_ax.set_xlim(0,1)
label_ax.set_ylim(0,1)
label_ax.axis("off")
label_ax.vlines(x=0.6, ymin=0.01, ymax=0.99, color="black")
label_ax.hlines(y=0.01, xmin=0.6, xmax=1, color="black")
label_ax.hlines(y=0.99, xmin=0.6, xmax=1, color="black")
label_ax.text(s="Epithelial pathway activity", x=0.5, y=0.5, fontsize=16, ha="right", va="center", rotation=90, weight="bold")
    

plt.savefig(f"{WORK_PATH}Figure/021_EcNP_cPosition_projeny_mat.png", dpi=500, format="png", bbox_inches=None)